# EXP001 division 诊断:分析 GT 分裂结构(纯 geff 读取)

In [ ]:
import os, subprocess, sys, glob
wheel_dir = '/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels'
targets = []
for pattern in ['zarr-3.*.whl', 'numcodecs-0.15*.whl', 'blosc2-*.whl', 'donfig-*.whl']:
    targets += sorted(glob.glob(wheel_dir + '/' + pattern))
print('installing wheels:')
for m in targets:
    print(' ', m.split('/')[-1])
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--force-reinstall', m],
                      capture_output=True, text=True, timeout=600)
    if r.returncode != 0:
        print('  FAIL:', (r.stdout or '')[-200:], (r.stderr or '')[-200:])
import zarr
print('zarr', zarr.__version__, 'ok')


In [ ]:
import numpy as np, pandas as pd
import zarr
from pathlib import Path
COMP_DIR = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
TRAIN_DIR = COMP_DIR / 'train'
CV8 = ['44b6_0113de3b','44b6_0b24845f','44b6_341df25f','44b6_e57ff5c6','6bba_05b6850b','6bba_05db0fb1','6bba_969618f6','6bba_fc83837d']
def read_geff(path):
    root = zarr.open(str(path), mode='r')
    node_ids = np.asarray(root['nodes/ids'])
    t = np.asarray(root['nodes/props/t/values'])
    z = np.asarray(root['nodes/props/z/values'])
    y = np.asarray(root['nodes/props/y/values'])
    x = np.asarray(root['nodes/props/x/values'])
    nodes = pd.DataFrame({'node_id': node_ids, 't': t, 'z': z, 'y': y, 'x': x})
    try:
        e = np.asarray(root['edges/ids'])
        edges = pd.DataFrame(e, columns=['source_id','target_id'])
    except Exception:
        edges = pd.DataFrame(columns=['source_id','target_id'])
    return nodes, edges
rows = []
for ds in CV8:
    gp = TRAIN_DIR / f'{ds}.geff'
    if not gp.exists():
        print('missing:', ds); continue
    try:
        nodes, edges = read_geff(gp)
    except Exception as e:
        print('read error', ds, type(e).__name__, str(e)[:200]); continue
    if edges.empty:
        print(ds, 'no edges'); continue
    src_counts = edges.groupby('source_id').size()
    div_parents = src_counts[src_counts == 2].index.tolist()
    for pid in div_parents:
        pnode = nodes[nodes.node_id == pid].iloc[0]
        children = edges[edges.source_id == pid]
        cinfo = []
        for cid in children.target_id:
            cn = nodes[nodes.node_id == cid].iloc[0]
            cinfo.append(dict(t=int(cn['t']), z=float(cn['z']), y=float(cn['y']), x=float(cn['x'])))
        rows.append(dict(dataset=ds, parent=int(pid), pt=int(pnode['t']),
                         pz=float(pnode['z']), py=float(pnode['y']), px=float(pnode['x']),
                         c1t=cinfo[0]['t'], c2t=cinfo[1]['t'], c1z=cinfo[0]['z'], c1y=cinfo[0]['y'], c1x=cinfo[0]['x'],
                         c2z=cinfo[1]['z'], c2y=cinfo[1]['y'], c2x=cinfo[1]['x']))
df = pd.DataFrame(rows)
print('total GT divisions across CV8:', len(df))
if len(df):
    print(df.to_string())
    df['dc1'] = df.c1t - df.pt; df['dc2'] = df.c2t - df.pt
    SCALE = np.array([1.625, 0.40625, 0.40625])
    def dd(rec, c):
        p = np.array([rec['pz'], rec['py'], rec['px']])*SCALE
        q = np.array([rec[c+'z'], rec[c+'y'], rec[c+'x']])*SCALE
        return float(np.linalg.norm(p-q))
    recs = df.to_dict('records')
    df['d_pc1'] = [dd(r,'c1') for r in recs]
    df['d_pc2'] = [dd(r,'c2') for r in recs]
    print('child time deltas:'); print(df[['dataset','pt','c1t','c2t','dc1','dc2']].to_string())
    print('parent->child dist (um):'); print(df[['dataset','d_pc1','d_pc2']].to_string())
    import collections
    all_dt = []
    for ds in CV8:
        gp = TRAIN_DIR / f'{ds}.geff'
        if not gp.exists(): continue
        nodes, edges = read_geff(gp)
        if edges.empty: continue
        tmap = dict(zip(nodes['node_id'], nodes['t']))
        for r in edges.itertuples(index=False):
            s, t = int(r.source_id), int(r.target_id)
            if s in tmap and t in tmap:
                all_dt.append(int(tmap[t]) - int(tmap[s]))
    print('all GT edge time-deltas:', collections.Counter(all_dt))
